# 06 — Stratégie & Dashboard (Person 6)

Brouillon d'exploration pour M7 (stratégie) et M8 (dashboard).
Croise `segment_profiles.csv`, `campaign_kpis.csv` et `churn_clv_predictions.csv`
pour produire des recommandations par segment.

Le code final et propre du dashboard est dans `dashboard/app.py`.
Ce notebook sert de brouillon d'exploration et documente le raisonnement.

In [ ]:
import pandas as pd

customers = pd.read_csv("../data/processed/customers_clean.csv")
segments = pd.read_csv("../data/processed/customer_segments.csv")
profiles = pd.read_csv("../data/processed/segment_profiles.csv")
campaigns = pd.read_csv("../data/processed/campaign_kpis.csv")
churn = pd.read_csv("../data/processed/churn_clv_predictions.csv")

profiles

## 1. Performance des campagnes par canal

In [ ]:
roi_by_channel = campaigns.groupby("Channel")["ROI"].mean().sort_values(ascending=False)
roi_by_channel

## 2. Churn moyen par segment (Cluster)

Attention : certains clients peuvent ne pas être classés (`Cluster` = `Unknown`)
s'ils sont absents de `customer_segments.csv` — à vérifier et signaler à Person 2.

In [ ]:
unclassified = churn[~churn["Customer_ID"].isin(segments["Customer_ID"])]
print(f"Clients non classés dans customer_segments.csv : {len(unclassified)}")
unclassified

In [ ]:
churn_classified = churn[churn["Customer_ID"].isin(segments["Customer_ID"])].copy()
churn_classified["Cluster"] = churn_classified["Cluster"].astype(float).astype(int)
churn_by_cluster = churn_classified.groupby("Cluster")["Churn_Probability"].mean()
churn_by_cluster

## 3. Tableau croisé : segment × canal préféré × ROI × churn

In [ ]:
strategy_table = profiles.copy()
strategy_table["ROI_Canal_Prefere"] = strategy_table["Preferred_Channel"].map(roi_by_channel)
strategy_table["Churn_Moyen"] = strategy_table["Cluster"].map(churn_by_cluster)

strategy_table[[
    "Cluster", "Persona_Name", "Size_pct", "Avg_Spend",
    "Preferred_Channel", "ROI_Canal_Prefere", "Churn_Moyen",
]]

## 4. Règle de priorisation (brouillon)

- Churn moyen > 15% ET taille importante → priorité rétention (🔴)
- ROI du canal préféré élevé (> 300%) ET churn faible → priorité investissement (🟢)
- Sinon → surveillance (🟡)

Client(s) non classé(s) avec churn élevé → anomalie à traiter en urgence, indépendamment
du raisonnement par segment (voir section 2).

In [ ]:
def recommend(row):
    if row["Churn_Moyen"] is not None and row["Churn_Moyen"] > 0.15:
        return "🔴 Priorité rétention"
    if row["ROI_Canal_Prefere"] is not None and row["ROI_Canal_Prefere"] > 300:
        return "🟢 Priorité investissement"
    return "🟡 Surveillance"

strategy_table["Priorite_M7"] = strategy_table.apply(recommend, axis=1)
strategy_table[["Persona_Name", "Priorite_M7"]]

## 5. Suite

- Rédaction complète des recommandations → `reports/M7_digital_strategy.pdf`
- Version interactive de ces mêmes analyses → `dashboard/app.py`
- Fusion avec le travail de toute l'équipe → `reports/final_report.docx` (M9)